---
title: "Orbital refinement for excited states of H4 using state-averaged DMRG"
author:
    - name: Thuy Truong
date: "2026-03-10"
categories: [code]
image: "H4_preview.png"
image-width: "1cm"
image-height: "1cm"
format:
    html:
        toc: true
        code-fold: false
code-annotations: hover
---

In this tutorial, we will explore the process of orbital refinement for excited states of the H4 molecule by using the NWChem orbitals as guess orbitals and performing state-averaged DMRG calculations. First, the parameters for the multiwavelet representation are defined.

In [ ]:
# | eval: false
molecule_name = "h4"
n_elec = 4  # <1>
number_roots = 3  # <2>
iterations = 3  # <3>
box_size = 50.0  # <4>
wavelet_order = 7  # <5>
madness_thresh = 0.0001  # <6>
basisset = "6-31g"  # <7>

1. Number of electrons
2. Number of states (ground state, 1st excited state, 2nd excited state)
3. Number of iterations for orbital refinement
4. Size of the simulation box
5. Order of wavelet basis functions
6. Threshold for numerical precision of function representation
7. Initial basis set for calculation

### Run NWChem calculation

To get the initial guess orbitals, we will perform a NWChem calculation. If you are using the FrayedEnds devcontainer or the singularity image, NWChem is already installed. Otherwise, you will need to install NWChem and **adjust the path** in the code below.

In [ ]:
# | eval: false
import subprocess as sp

nwchem_input = (
    """
title "molecule"
memory stack 1500 mb heap 100 mb global 1400 mb
charge 0
geometry units angstrom noautosym nocenter
    H 0.0 0.0 -1.5
    H 0.0 0.0 -0.5
    H 0.0 0.0 0.5
    H 0.0 0.0 1.5
end
basis
  * library """
    + basisset
    + """
end
scf
 maxiter 200
end
task scf
"""
)

with open("nwchem", "w") as f:  # <1>
    f.write(nwchem_input)  # <1>
programm = sp.call(
    "/opt/anaconda3/envs/frayedends/bin/nwchem nwchem",  # <2>
    stdout=open("nwchem.out", "w"),
    stderr=open("nwchem_err.log", "w"),
    shell=True,
)  # <1>

1. Run NWChem calculation
2. Adjust the path to NWChem here if you are not using the FrayedEnds devcontainer or the singularity image.

### Convert NWChem AOs and MOs to MRA-Orbitals
Next, we will read the molecular orbitals (MOs) from the NWChem calculation and translate them into multiwavelets by using the NWChem_Converter class in FrayedEnds.

In [ ]:
# | eval: false
import frayedends as fe

world = fe.MadWorld3D(L=box_size, k=wavelet_order, thresh=madness_thresh)  # <1>

converter = fe.NWChem_Converter(world)  # <2>
converter.read_nwchem_file("nwchem")  # <3>
orbs = converter.get_mos()  # <4>
Vnuc = converter.get_Vnuc()  # <5>
nuclear_repulsion_energy = converter.get_nuclear_repulsion_energy()  # <6>

n_orbitals = len(orbs)

molecule = fe.MolecularGeometry(units="angstrom")  # <7>
molecule.add_atom(0.0, 0.0, -1.5, "H")  # <7>
molecule.add_atom(0.0, 0.0, -0.5, "H")  # <7>
molecule.add_atom(0.0, 0.0, 0.5, "H")  # <7>
molecule.add_atom(0.0, 0.0, 1.5, "H")  # <7>

for i in range(n_orbitals):
    world.cube_plot(f"initial_orb{i}", orbs[i], molecule)

1. Setting up the numerical environment for the MRA calculations by creating a frayedends world object with the specified parameters
2. Create an NWChem_Converter object
3. Read the NWChem output file to extract the orbitals and other relevant information
4. Get the molecular orbitals (MOs) from the converter
5. Get the nuclear potential from the converter
6. Get the nuclear repulsion energy from the converter
7. Define a linear H4 molecule geometry with 1.0 Angstrom spacing between adjacent atoms

`MADNESS runtime initialized with 9 threads in the pool and affinity OFF`

### Calculate initial integrals

After obtaining the orbitals, we can calculate the initial integrals required for the DMRG calculation, including the two-body, kinetic, potential, and overlap integrals.

In [ ]:
# | eval: false
integrals = fe.Integrals3D(world)  # <1>
G = integrals.compute_two_body_integrals(orbs, ordering="chem").elems  # <2>
T = integrals.compute_kinetic_integrals(orbs)  # <2>
V = integrals.compute_potential_integrals(orbs, Vnuc)  # <2>
h1 = T + V # <2>
S = integrals.compute_overlap_integrals(orbs)  # <2>

1. Create an integrals object to compute the initial integrals
2. Compute the two-body, kinetic, potential, and overlap integrals using the orbitals obtained from NWChem

### Perform state-averaged DMRG calculation with orbital reordering and extract RDMs

To improve the efficiency of the DMRG calculation, an orbital reordering is performed using the one-body integral $T + V$ and the two-body integral $G$ obtained from the previous step.
We then perform a state-averaged DMRG calculation using the reordered integrals and extract the one-body and two-body reduced density matrices (rdms). Finally, the resulting RDMs are transformed back into the original orbital ordering.

In [ ]:
# | eval: false
import numpy as np
from pyblock2.driver.core import DMRGDriver, SymmetryTypes

driver = DMRGDriver(scratch="./tmp", symm_type=SymmetryTypes.SU2, n_threads=8)  # <1>
driver.initialize_system(n_sites=n_orbitals, n_elec=n_elec, spin=0)  # <1>
mpo = driver.get_qc_mpo(h1e=h1, g2e=G, ecore=nuclear_repulsion_energy, iprint=0)  # <1>
ket = driver.get_random_mps(tag="KET", bond_dim=100, nroots=number_roots)  # <1>
energies = driver.dmrg(
    mpo, ket, n_sweeps=10, bond_dims=[100], noises=[1e-5] * 4 + [0], thrds=[1e-10] * 8, iprint=1
)  # <1>

idx = driver.orbital_reordering(h1, G) # <2>
h1_new = h1[idx][:, idx] # <2>
g2_new = G[idx][:, idx][:, :, idx][:, :, :, idx] # <2>

driver.initialize_system(n_sites=n_orbitals, n_elec=n_elec, spin=0) # <3>
mpo = driver.get_qc_mpo(h1e=h1_new, g2e=g2_new, ecore=nuclear_repulsion_energy, iprint=0) # <3>
ket = driver.get_random_mps(tag="KET", bond_dim=100, nroots=number_roots) # <3>
energies = driver.dmrg(mpo, ket, n_sweeps=10, bond_dims=[100], noises=[1e-5] * 4 + [0], thrds=[1e-10] * 8, iprint=1) # <3>
print("State-averaged MPS energies = [%s]" % " ".join("%20.15f" % x for x in energies))  # <3>

kets = [driver.split_mps(ket, ir, tag="KET-%d" % ir) for ir in range(ket.nroots)] # <4>
sa_1pdm = np.mean([driver.get_1pdm(k) for k in kets], axis=0)  # <5>
sa_2pdm = np.mean([driver.get_2pdm(k) for k in kets], axis=0).transpose(0, 3, 1, 2)  # <6>
print(
    "Energy from SA-pdms = %20.15f"
    % (np.einsum("ij,ij->", sa_1pdm, h1_new) + 0.5 * np.einsum("ijkl,ijkl->", sa_2pdm, g2_new) + nuclear_repulsion_energy)
)

idx_back = np.zeros(len(idx), dtype=int) # <7>
for i in range(len(idx)): # <7>
    idx_back[idx[i]] = i # <7>

sa_1pdm = sa_1pdm[idx_back][:, idx_back] # <7>
sa_2pdm = sa_2pdm[idx_back][:, idx_back][:, :, idx_back][:, :, :, idx_back] # <7>
sa_2pdm_phys = sa_2pdm.swapaxes(1, 2)

1. Perform State Average (SA) DMRG calculation
2. Perform an orbital reordering using the one-body and two-body integrals
3. Perform a second State Average (SA) DMRG calculation with the reordered integrals
4. Extract reduced density matrices (rdms)
5. Compute the state-average one-body reduced density matrix
6. Compute the state average two-body reduced density matrix
7. Transform the rdms back into the original orbital ordering

```
Sweep =    0 | Direction =  forward | Bond dimension =  100 | Noise =  1.00e-05 | Dav threshold =  1.00e-10
Time elapsed =      1.163 | E[  3] =      -2.2248639286     -1.8539249222     -1.8307814235 | DW = 5.67495e-20

Sweep =    1 | Direction = backward | Bond dimension =  100 | Noise =  1.00e-05 | Dav threshold =  1.00e-10
Time elapsed =      1.586 | E[  3] =      -2.2248639286     -1.8539249222     -1.8307814235 | DE = 1.78e-15 | DW = 5.89849e-20

Sweep =    2 | Direction =  forward | Bond dimension =  100 | Noise =  1.00e-05 | Dav threshold =  1.00e-10
Time elapsed =      1.995 | E[  3] =      -2.2248639286     -1.8539249222     -1.8307814235 | DE = 1.78e-15 | DW = 2.32726e-19

Sweep =    3 | Direction = backward | Bond dimension =  100 | Noise =  1.00e-05 | Dav threshold =  1.00e-10
Time elapsed =      2.524 | E[  3] =      -2.2248639286     -1.8539249222     -1.8307814235 | DE = 8.88e-16 | DW = 5.23044e-20

Sweep =    4 | Direction =  forward | Bond dimension =  100 | Noise =  0.00e+00 | Dav threshold =  1.00e-10
Time elapsed =      2.935 | E[  3] =      -2.2248639286     -1.8539249222     -1.8307814235 | DE = 8.88e-16 | DW = 2.53627e-20


Sweep =    0 | Direction =  forward | Bond dimension =  100 | Noise =  1.00e-05 | Dav threshold =  1.00e-10
Time elapsed =      0.367 | E[  3] =      -2.2248639286     -1.8539249223     -1.8307814236 | DW = 4.66699e-19

Sweep =    1 | Direction = backward | Bond dimension =  100 | Noise =  1.00e-05 | Dav threshold =  1.00e-10
Time elapsed =      0.709 | E[  3] =      -2.2248639286     -1.8539249223     -1.8307814236 | DE = 2.66e-15 | DW = 4.26176e-20

Sweep =    2 | Direction =  forward | Bond dimension =  100 | Noise =  1.00e-05 | Dav threshold =  1.00e-10
Time elapsed =      0.959 | E[  3] =      -2.2248639286     -1.8539249223     -1.8307814236 | DE = -4.44e-15 | DW = 7.54028e-19

Sweep =    3 | Direction = backward | Bond dimension =  100 | Noise =  1.00e-05 | Dav threshold =  1.00e-10
Time elapsed =      1.218 | E[  3] =      -2.2248639286     -1.8539249223     -1.8307814236 | DE = -4.44e-15 | DW = 4.47832e-20

Sweep =    4 | Direction =  forward | Bond dimension =  100 | Noise =  0.00e+00 | Dav threshold =  1.00e-10
Time elapsed =      1.286 | E[  3] =      -2.2248639286     -1.8539249223     -1.8307814236 | DE = 0.00e+00 | DW = 2.09600e-20

State-averaged MPS energies = [  -2.224863928635028   -1.853924922268516   -1.830781423564431]
Energy from SA-pdms =   -1.969856758155991
```

### Orbital refinement

Finally, we will perform the orbital refinement by using the state-averaged 1-body and 2-body reduced density matrices obtained from the DMRG calculation. The refined orbitals can then be used for further state-averaged DMRG calculations to improve the accuracy of the excited state energies. The orbital refinement is repeated for a specified number of iterations (here: 3), as defined at the beginning of the tutorial.

In [ ]:
# | eval: false
import time

for iter in range(iterations):
    iter_start = time.perf_counter()

    opti = fe.Optimization3D(world, Vnuc, nuclear_repulsion_energy)  # <1>
    orbs = opti.get_orbitals(orbitals=orbs, rdm1=sa_1pdm, rdm2=sa_2pdm_phys, opt_thresh=0.001, occ_thresh=0.001)  # <1>

    for i in range(n_orbitals):
        world.cube_plot(f"orb{i}", orbs[i], molecule)  # <2>

    G = integrals.compute_two_body_integrals(orbs, ordering="chem").elems  # <3>
    T = integrals.compute_kinetic_integrals(orbs)  # <3>
    V = integrals.compute_potential_integrals(orbs, Vnuc)  # <3>
    h1 = T + V # <3>
    S = integrals.compute_overlap_integrals(orbs)  # <3>

    driver = DMRGDriver(scratch="./tmp", symm_type=SymmetryTypes.SU2, n_threads=8)  # <4>
    driver.initialize_system(n_sites=n_orbitals, n_elec=n_elec, spin=0)  # <4>
    mpo = driver.get_qc_mpo(h1e=h1, g2e=G, ecore=nuclear_repulsion_energy, iprint=0)  # <4>
    ket = driver.get_random_mps(tag="KET", bond_dim=100, nroots=number_roots)  # <4>
    energies = driver.dmrg(
        mpo, ket, n_sweeps=10, bond_dims=[100], noises=[1e-5] * 4 + [0], thrds=[1e-10] * 8, iprint=1
    )  # <4>

    idx = driver.orbital_reordering(h1, G) # <5>
    h1_new = h1[idx][:, idx] # <5>
    g2_new = G[idx][:, idx][:, :, idx][:, :, :, idx] # <5>

    driver.initialize_system(n_sites=n_orbitals, n_elec=n_elec, spin=0) # <6>
    mpo = driver.get_qc_mpo(h1e=h1_new, g2e=g2_new, ecore=nuclear_repulsion_energy, iprint=0) # <6>
    ket = driver.get_random_mps(tag="KET", bond_dim=100, nroots=number_roots) # <6>
    energies = driver.dmrg(mpo, ket, n_sweeps=10, bond_dims=[100], noises=[1e-5] * 4 + [0], thrds=[1e-10] * 8, iprint=1) # <6>
    print("State-averaged MPS energies after refinement = [%s]" % " ".join("%20.15f" % x for x in energies))

    kets = [driver.split_mps(ket, ir, tag="KET-%d" % ir) for ir in range(ket.nroots)] # <7>
    sa_1pdm = np.mean([driver.get_1pdm(k) for k in kets], axis=0)  # <8>
    sa_2pdm = np.mean([driver.get_2pdm(k) for k in kets], axis=0).transpose(0, 3, 1, 2)  # <9>
    print(
        "Energy from SA-pdms = %20.15f"
        % (np.einsum("ij,ij->", sa_1pdm, h1_new) + 0.5 * np.einsum("ijkl,ijkl->", sa_2pdm, g2_new) + nuclear_repulsion_energy)
    )
    idx_back = np.zeros(len(idx), dtype=int) # <10>
    for i in range(len(idx)): # <10>
        idx_back[idx[i]] = i # <10>

    sa_1pdm = sa_1pdm[idx_back][:, idx_back] # <10>
    sa_2pdm = sa_2pdm[idx_back][:, idx_back][:, :, idx_back][:, :, :, idx_back] # <10>
    sa_2pdm_phys = sa_2pdm.swapaxes(1, 2)

fe.cleanup(globals())

1. Create an optimization object for orbital refinement and get the refined orbitals using the state-averaged one-body and two-body rdms
2. Plot the refined orbitals
3. Calculate the integrals with the refined orbitals
4. Perform a state-averaged DMRG calculation with the refined orbitals
5. Perform an orbital reordering using the one-body and two-body integrals
6. Perform a second state-averaged DMRG calculation with the reordered integrals
7. Extract reduced density matrices (rdms)
8. Compute the state-average one-body rdm with the refined orbitals
9. Compute the state-average two-body rdm with the refined orbitals
10. Transform the rdms back into the original orbital ordering

<details>
<summary>Output from the orbital refinement loop</summary>

<pre><code>

Sweep =    0 | Direction =  forward | Bond dimension =  100 | Noise =  1.00e-05 | Dav threshold =  1.00e-10
Time elapsed =      1.486 | E[  3] =      -2.2370817416     -1.9070522959     -1.8537964808 | DW = 1.19185e-19

Sweep =    1 | Direction = backward | Bond dimension =  100 | Noise =  1.00e-05 | Dav threshold =  1.00e-10
Time elapsed =      1.914 | E[  3] =      -2.2370817416     -1.9070522959     -1.8537964808 | DE = -8.88e-16 | DW = 5.21074e-20

Sweep =    2 | Direction =  forward | Bond dimension =  100 | Noise =  1.00e-05 | Dav threshold =  1.00e-10
Time elapsed =      2.206 | E[  3] =      -2.2370817416     -1.9070522959     -1.8537964808 | DE = -9.77e-15 | DW = 1.08084e-19

Sweep =    3 | Direction = backward | Bond dimension =  100 | Noise =  1.00e-05 | Dav threshold =  1.00e-10
Time elapsed =      2.630 | E[  3] =      -2.2370817416     -1.9070522959     -1.8537964808 | DE = 8.88e-16 | DW = 6.40138e-20

Sweep =    4 | Direction =  forward | Bond dimension =  100 | Noise =  0.00e+00 | Dav threshold =  1.00e-10
Time elapsed =      2.814 | E[  3] =      -2.2370817416     -1.9070522959     -1.8537964808 | DE = 3.55e-15 | DW = 4.57325e-20


Sweep =    0 | Direction =  forward | Bond dimension =  100 | Noise =  1.00e-05 | Dav threshold =  1.00e-10
Time elapsed =      1.612 | E[  3] =      -2.2370817416     -1.9070522959     -1.8537964808 | DW = 1.84445e-19

Sweep =    1 | Direction = backward | Bond dimension =  100 | Noise =  1.00e-05 | Dav threshold =  1.00e-10
Time elapsed =      1.891 | E[  3] =      -2.2370817416     -1.9070522959     -1.8537964808 | DE = -8.88e-16 | DW = 2.78420e-20

Sweep =    2 | Direction =  forward | Bond dimension =  100 | Noise =  1.00e-05 | Dav threshold =  1.00e-10
Time elapsed =      2.100 | E[  3] =      -2.2370817416     -1.9070522959     -1.8537964808 | DE = 8.88e-16 | DW = 2.04572e-19

Sweep =    3 | Direction = backward | Bond dimension =  100 | Noise =  1.00e-05 | Dav threshold =  1.00e-10
Time elapsed =      2.466 | E[  3] =      -2.2370817416     -1.9070522959     -1.8537964808 | DE = -8.88e-15 | DW = 8.63092e-20

Sweep =    4 | Direction =  forward | Bond dimension =  100 | Noise =  0.00e+00 | Dav threshold =  1.00e-10
Time elapsed =      2.601 | E[  3] =      -2.2370817416     -1.9070522959     -1.8537964808 | DE = 8.88e-16 | DW = 2.21081e-20

State-averaged MPS energies after refinement = [  -2.237081741646887   -1.907052295896205   -1.853796480842818]
Energy from SA-pdms =   -1.999310172795306

Sweep =    0 | Direction =  forward | Bond dimension =  100 | Noise =  1.00e-05 | Dav threshold =  1.00e-10
Time elapsed =      1.557 | E[  3] =      -2.2358492873     -1.9227693728     -1.8543687643 | DW = 1.16552e-19

Sweep =    1 | Direction = backward | Bond dimension =  100 | Noise =  1.00e-05 | Dav threshold =  1.00e-10
Time elapsed =      1.924 | E[  3] =      -2.2358492873     -1.9227693728     -1.8543687643 | DE = -8.88e-16 | DW = 4.99879e-20

Sweep =    2 | Direction =  forward | Bond dimension =  100 | Noise =  1.00e-05 | Dav threshold =  1.00e-10
Time elapsed =      2.280 | E[  3] =      -2.2358492873     -1.9227693728     -1.8543687643 | DE = 4.44e-15 | DW = 1.10977e-19

Sweep =    3 | Direction = backward | Bond dimension =  100 | Noise =  1.00e-05 | Dav threshold =  1.00e-10
Time elapsed =      2.721 | E[  3] =      -2.2358492873     -1.9227693728     -1.8543687643 | DE = -8.88e-16 | DW = 6.15774e-20

Sweep =    4 | Direction =  forward | Bond dimension =  100 | Noise =  0.00e+00 | Dav threshold =  1.00e-10
Time elapsed =      3.146 | E[  3] =      -2.2358492873     -1.9227693728     -1.8543687643 | DE = -7.99e-15 | DW = 3.35582e-20


Sweep =    0 | Direction =  forward | Bond dimension =  100 | Noise =  1.00e-05 | Dav threshold =  1.00e-10
Time elapsed =      2.093 | E[  3] =      -2.2358492873     -1.9227693727     -1.8543687643 | DW = 2.38636e-19

Sweep =    1 | Direction = backward | Bond dimension =  100 | Noise =  1.00e-05 | Dav threshold =  1.00e-10
Time elapsed =      2.386 | E[  3] =      -2.2358492873     -1.9227693727     -1.8543687643 | DE = 5.33e-15 | DW = 1.36163e-19

Sweep =    2 | Direction =  forward | Bond dimension =  100 | Noise =  1.00e-05 | Dav threshold =  1.00e-10
Time elapsed =      2.830 | E[  3] =      -2.2358492873     -1.9227693727     -1.8543687643 | DE = 0.00e+00 | DW = 1.21338e-19

Sweep =    3 | Direction = backward | Bond dimension =  100 | Noise =  1.00e-05 | Dav threshold =  1.00e-10
Time elapsed =      3.113 | E[  3] =      -2.2358492873     -1.9227693727     -1.8543687643 | DE = 0.00e+00 | DW = 3.78870e-19

Sweep =    4 | Direction =  forward | Bond dimension =  100 | Noise =  0.00e+00 | Dav threshold =  1.00e-10
Time elapsed =      3.411 | E[  3] =      -2.2358492873     -1.9227693727     -1.8543687643 | DE = -1.78e-15 | DW = 4.08407e-20

State-averaged MPS energies after refinement = [  -2.235849287317252   -1.922769372728945   -1.854368764314865]
Energy from SA-pdms =   -2.004329141453686

Sweep =    0 | Direction =  forward | Bond dimension =  100 | Noise =  1.00e-05 | Dav threshold =  1.00e-10
Time elapsed =      2.174 | E[  3] =      -2.2348149044     -1.9295230913     -1.8545233285 | DW = 1.80167e-19

Sweep =    1 | Direction = backward | Bond dimension =  100 | Noise =  1.00e-05 | Dav threshold =  1.00e-10
Time elapsed =      2.375 | E[  3] =      -2.2348149044     -1.9295230913     -1.8545233285 | DE = 2.66e-15 | DW = 4.85680e-20

Sweep =    2 | Direction =  forward | Bond dimension =  100 | Noise =  1.00e-05 | Dav threshold =  1.00e-10
Time elapsed =      2.930 | E[  3] =      -2.2348149044     -1.9295230913     -1.8545233285 | DE = 5.33e-15 | DW = 1.11411e-19

Sweep =    3 | Direction = backward | Bond dimension =  100 | Noise =  1.00e-05 | Dav threshold =  1.00e-10
Time elapsed =      3.440 | E[  3] =      -2.2348149044     -1.9295230913     -1.8545233285 | DE = 0.00e+00 | DW = 7.07280e-20

Sweep =    4 | Direction =  forward | Bond dimension =  100 | Noise =  0.00e+00 | Dav threshold =  1.00e-10
Time elapsed =      3.943 | E[  3] =      -2.2348149044     -1.9295230913     -1.8545233285 | DE = -8.88e-16 | DW = 3.45844e-20


Sweep =    0 | Direction =  forward | Bond dimension =  100 | Noise =  1.00e-05 | Dav threshold =  1.00e-10
Time elapsed =      1.383 | E[  3] =      -2.2348149044     -1.9295230913     -1.8545233285 | DW = 1.14703e-19

Sweep =    1 | Direction = backward | Bond dimension =  100 | Noise =  1.00e-05 | Dav threshold =  1.00e-10
Time elapsed =      1.670 | E[  3] =      -2.2348149044     -1.9295230913     -1.8545233285 | DE = 1.24e-14 | DW = 1.14906e-18

Sweep =    2 | Direction =  forward | Bond dimension =  100 | Noise =  1.00e-05 | Dav threshold =  1.00e-10
Time elapsed =      2.166 | E[  3] =      -2.2348149044     -1.9295230913     -1.8545233285 | DE = 8.88e-16 | DW = 2.78132e-19

Sweep =    3 | Direction = backward | Bond dimension =  100 | Noise =  1.00e-05 | Dav threshold =  1.00e-10
Time elapsed =      2.459 | E[  3] =      -2.2348149044     -1.9295230913     -1.8545233285 | DE = -8.88e-16 | DW = 3.43592e-19

Sweep =    4 | Direction =  forward | Bond dimension =  100 | Noise =  0.00e+00 | Dav threshold =  1.00e-10
Time elapsed =      2.948 | E[  3] =      -2.2348149044     -1.9295230913     -1.8545233285 | DE = 8.88e-16 | DW = 2.10052e-20

State-averaged MPS energies after refinement = [  -2.234814904414891   -1.929523091285816   -1.854523328461209]
Energy from SA-pdms =   -2.006287108053977

</code></pre>
</details>

The orbital refinement process is repeated for the specified number of iterations.
At the end of the orbital refinement, the energies of the ground state and excited states should be improved compared to the initial DMRG calculation with the NWChem orbitals as guess orbitals.

The initial orbitals obtained from NWChem and the refined orbitals after the orbital optimization can be visualized using the cube files and plotted **interactively** using py3Dmol.


In [ ]:
# | echo: false

import py3Dmol

initial_orb = open("initial_orb1.cube").read()
refined_orb = open("orb1.cube").read()

v = py3Dmol.view(width=800, height=400, viewergrid=(1, 2))

v.addVolumetricData(initial_orb, "cube", {"isoval": -0.001, "color": "red", "opacity": 0.75}, viewer=(0, 0))
v.addVolumetricData(initial_orb, "cube", {"isoval": 0.001, "color": "blue", "opacity": 0.75}, viewer=(0, 0))
v.addModel(initial_orb, "cube", viewer=(0, 0))
v.setStyle({"sphere": {}}, viewer=(0, 0))

v.addVolumetricData(refined_orb, "cube", {"isoval": -0.001, "color": "red", "opacity": 0.75}, viewer=(0, 1))
v.addVolumetricData(refined_orb, "cube", {"isoval": 0.001, "color": "blue", "opacity": 0.75}, viewer=(0, 1))
v.addModel(refined_orb, "cube", viewer=(0, 1))
v.setStyle({"sphere": {}}, viewer=(0, 1))

v.zoomTo()
v.show()